# Modelling — NYC Rideshare Earnings

Trains a Linear Regression baseline and a Lasso (L1) regularised model
on May–October 2023 trips, then evaluates against the November 2023
temporal holdout. All ML logic lives in `scripts/modelling.py`.

In [ ]:
import sys
sys.path.append('../scripts')

from spark import get_spark
from io_utils import load_curated
from modelling import (
    assemble_features, temporal_split, train_linear_regression, evaluate,
)

spark = get_spark(executor_memory='12g', shuffle_partitions=400)

## 1. Load Curated Data & Assemble Features

Loads the curated month-partitioned Parquet under a fixed schema,
then combines license / time / location / weather / holiday features
into a single `features` vector for ML.

In [ ]:
final_df = load_curated(spark)
final_data = assemble_features(final_df)
final_data.limit(5).show(truncate=False)

## 2. Temporal Train/Test Split

Train on months 5–10 (May–October 2023, ~90M trips); hold out month 11
(November 2023, ~15M trips) as the test set. A temporal split avoids
the leakage that random shuffling would introduce, and mirrors how the
model would be evaluated against genuinely future months.

In [ ]:
training_data, test_data = temporal_split(final_data)
print(f'Training set: {training_data.count():,}')
print(f'Test set:     {test_data.count():,}')

## 3. Baseline: Linear Regression

In [ ]:
linear_model = train_linear_regression(training_data)
print('Intercept:', linear_model.intercept)

linear_predictions = linear_model.transform(test_data)
print(evaluate(linear_predictions))

linear_predictions.select('earnings', 'prediction').write.mode('overwrite').parquet(
    '../data/results/linear_pred.parquet'
)
test_data.select('earnings').write.mode('overwrite').parquet(
    '../data/results/actual_values.parquet'
)

## 4. Lasso Regression (L1, λ=1)

`elasticNetParam=1.0` gives pure L1 regularisation, which shrinks
less informative coefficients toward zero. `regParam=1` is a moderate
strength — higher values increase sparsity at the cost of bias.

In [ ]:
lasso_model = train_linear_regression(
    training_data, reg_param=1.0, elastic_net_param=1.0,
)
lasso_predictions = lasso_model.transform(test_data)
print(evaluate(lasso_predictions))

lasso_predictions.select('earnings', 'prediction').write.mode('overwrite').parquet(
    '../data/results/regularised_linear_pred.parquet'
)